# LangChain + ChromaDB: RAG básico

El objetivo es construir un **RAG básico** usando:

- **OpenAI API** para chat y embeddings.
- **LangChain** con LCEL / Runnables.
- **ChromaDB** como vector store local persistente.

> Idea central: un LLM no conoce automáticamente tus documentos privados. RAG permite buscar fragmentos relevantes en una base vectorial y pasarlos al modelo como contexto.


## 0. Qué vamos a construir

Al final del notebook tendremos esta arquitectura:

```text
Documentos -> chunks -> embeddings -> ChromaDB
                                    |
Pregunta del usuario -> retriever --+
                                    |
                          prompt con contexto -> ChatOpenAI -> respuesta
```

El ejemplo usa documentos creados dentro del notebook para que funcione en clase sin depender de archivos externos. Luego se agregan variantes para usar PDFs o textos propios.


## 1. Instalación

Ejecutar esta celda una vez en un entorno limpio.

Notas:

- `langchain` contiene la capa principal moderna.
- `langchain-openai` contiene la integración con OpenAI.
- `langchain-chroma` contiene la integración actual entre LangChain y Chroma.
- `chromadb` es la base vectorial.
- `langchain-text-splitters` contiene splitters de documentos.
- `pypdf` se usa solo si querés cargar PDFs más adelante.


In [ ]:
%pip install -U     langchain     langchain-openai     langchain-chroma     chromadb     langchain-text-splitters     python-dotenv     pypdf     tiktoken


## 3. API key y modelos

No conviene hardcodear claves en notebooks que se comparten con alumnos. Usamos variables de entorno o un archivo `.env`.

Ejemplo de `.env`:

```text
OPENAI_API_KEY=tu_api_key
OPENAI_MODEL=gpt-5.4-mini
OPENAI_EMBEDDING_MODEL=text-embedding-3-small
```

Para clase conviene usar un modelo de bajo costo/latencia para chat y `text-embedding-3-small` para embeddings.


In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-nano-2026-03-17")
OPENAI_EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

print("Chat model:", OPENAI_MODEL)
print("Embedding model:", OPENAI_EMBEDDING_MODEL)


## 4. Primer objeto LangChain: chat model

En LangChain, `ChatOpenAI` envuelve un modelo conversacional. La llamada básica se hace con `.invoke()`.


In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=OPENAI_MODEL,
    temperature=0.2,
)

respuesta = llm.invoke("Explicá RAG en una frase simple para estudiantes de programación.")
print(respuesta.content)


## 5. Mini-chain con LCEL

LCEL permite componer piezas con `|`:

```text
prompt -> modelo -> parser
```

Esto reemplaza gran parte del uso clásico de `LLMChain` para casos simples.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt_intro = ChatPromptTemplate.from_messages([
    ("system", "Sos un profesor claro y práctico."),
    ("human", "Explicá el concepto de {tema} con una analogía simple."),
])

chain_intro = prompt_intro | llm | StrOutputParser()

print(chain_intro.invoke({"tema": "embeddings"}))


## 6. Documentos de ejemplo

Para entender RAG no necesitamos empezar con PDFs. Primero creamos documentos con metadatos.

Cada `Document` tiene:

- `page_content`: texto.
- `metadata`: información útil para filtrar o citar la fuente.


In [ ]:
from langchain_core.documents import Document

docs = [
    Document(
        page_content=(
            "La empresa FicticiaBot ofrece tres planes: Básico, Pro y Enterprise. "
            "El plan Básico incluye 1.000 consultas mensuales y soporte por email."
        ),
        metadata={"source": "politicas_planes.md", "section": "planes"},
    ),
    Document(
        page_content=(
            "El plan Pro incluye 20.000 consultas mensuales, soporte prioritario y acceso a métricas. "
            "Tiene un límite de 5 usuarios por workspace."
        ),
        metadata={"source": "politicas_planes.md", "section": "planes"},
    ),
    Document(
        page_content=(
            "El plan Enterprise incluye consultas personalizadas, SSO, auditoría avanzada y soporte dedicado. "
            "Los límites se acuerdan por contrato."
        ),
        metadata={"source": "politicas_enterprise.md", "section": "enterprise"},
    ),
    Document(
        page_content=(
            "La política de privacidad establece que los datos de clientes no se usan para entrenar modelos externos. "
            "Los logs se retienen durante 30 días salvo acuerdo contractual diferente."
        ),
        metadata={"source": "privacidad.md", "section": "datos"},
    ),
    Document(
        page_content=(
            "Para solicitar baja del servicio, el cliente debe enviar una solicitud desde el panel de administración. "
            "La baja se procesa dentro de los 5 días hábiles."
        ),
        metadata={"source": "soporte.md", "section": "bajas"},
    ),
]

print("Cantidad de documentos:", len(docs))
print(docs[0])


## 7. Split de documentos en chunks

Los modelos tienen una ventana de contexto limitada y los documentos reales suelen ser largos. Por eso se dividen en fragmentos.

Parámetros importantes:

- `chunk_size`: tamaño aproximado del fragmento.
- `chunk_overlap`: solapamiento entre fragmentos para no cortar ideas importantes.

En producción, estos valores se ajustan según el tipo de documento y la calidad de recuperación.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=60,
)

splits = splitter.split_documents(docs)

print("Chunks generados:", len(splits))
for i, chunk in enumerate(splits[:3], start=1):
    print("--- chunk", i)
    print(chunk.page_content)
    print(chunk.metadata)




## 8. Embeddings

Un embedding convierte texto en un vector numérico. Textos semánticamente parecidos quedan cerca en el espacio vectorial.

En RAG se usan embeddings para buscar los chunks más parecidos a la pregunta del usuario.


In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model=OPENAI_EMBEDDING_MODEL)

vector = embeddings.embed_query("¿Qué incluye el plan Pro?")

print("Dimensión del vector:", len(vector))
print("Primeros 5 valores:", vector)



## 9. Crear una base vectorial local con Chroma

Usamos `Chroma` desde `langchain_chroma`, que es el paquete de integración actual.



In [ ]:
import shutil
from pathlib import Path
from langchain_chroma import Chroma

CHROMA_DIR = "./chroma_rag_demo"
COLLECTION_NAME = "curso_rag_basico"

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_DIR,
)

print("Vector store creado en:", Path(CHROMA_DIR).resolve())
print("Cantidad aproximada de chunks:", vectorstore._collection.count())

## 10. Búsqueda semántica simple

Antes de usar un LLM, probamos si el buscador encuentra chunks relevantes.


In [ ]:
query = "¿Cuántos usuarios permite el plan Pro?"
results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results, start=1):
    print(f"Resultado {i}")
    print("Fuente:", doc.metadata)
    print(doc.page_content)
    print()


## 11. Convertir Chroma en retriever

Un `retriever` recibe una pregunta y devuelve documentos relevantes. Es la interfaz que normalmente conectamos a una chain de RAG.


In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

retrieved_docs = retriever.invoke("¿Qué pasa con los datos de clientes?")

for doc in retrieved_docs:
    print(doc.metadata, "->", doc.page_content[:120], "...")


## 12. RAG con LCEL

Ahora conectamos:

1. Pregunta del usuario.
2. Retriever.
3. Prompt con contexto.
4. Modelo.
5. Parser de salida.

El prompt le pide explícitamente al modelo que no invente si la respuesta no está en el contexto.


In [ ]:
from langchain_core.runnables import RunnablePassthrough


def format_docs(documents):
    return "\n\n".join(
        f"Fuente: {doc.metadata.get('source')} | Sección: {doc.metadata.get('section')}\n{doc.page_content}"
        for doc in documents
    )

rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Sos un asistente de soporte. Respondé usando únicamente el contexto dado. "
        "Si la respuesta no está en el contexto, decí: 'No encuentro esa información en la base de conocimiento'.",
    ),
    (
        "human",
        "Contexto:\n{context}\n\nPregunta: {question}",
    ),
])

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke("¿Qué incluye el plan Enterprise?"))


## 13. RAG devolviendo respuesta + fuentes

En muchas apps no alcanza con responder: también queremos mostrar qué documentos fueron usados.

Para eso separamos la recuperación de la generación.


In [ ]:
def answer_with_sources(question: str):
    retrieved = retriever.invoke(question)
    context = format_docs(retrieved)
    answer = (rag_prompt | llm | StrOutputParser()).invoke({
        "context": context,
        "question": question,
    })
    sources = []
    for doc in retrieved:
        item = {
            "source": doc.metadata.get("source"),
            "section": doc.metadata.get("section"),
        }
        if item not in sources:
            sources.append(item)
    return {"answer": answer, "sources": sources}

result = answer_with_sources("¿Cuánto tardan en procesar la baja del servicio?")
print(result["answer"])
print("\nFuentes:")
for source in result["sources"]:
    print("-", source)


## 14. Pregunta fuera de la base

Una buena demo de RAG debe mostrar qué pasa cuando la información no está en los documentos.


In [ ]:
print(rag_chain.invoke("¿Cuál es el precio exacto del plan Pro en dólares?"))


## 15. Filtros por metadata

Chroma permite buscar usando metadatos. Esto es útil cuando tenemos secciones, clientes, fechas, permisos o tipos de documento.

Ejemplo: buscar solamente en documentos de la sección `planes`.


In [ ]:
results_filtered = vectorstore.similarity_search(
    "¿Qué incluye Enterprise?",
    k=3,
    filter={"section": "planes"},
)

for doc in results_filtered:
    print(doc.metadata)
    print(doc.page_content)
    print()


## 16. Cargar PDFs propios (opcional)

> Ejecutar solo si tenés archivos en la carpeta `data/`.


In [ ]:
# Opcional: cargar PDFs desde una carpeta data/
# from langchain_community.document_loaders import PyPDFLoader
# from pathlib import Path
#
# pdf_docs = []
# for pdf_path in Path("data").glob("*.pdf"):
#     loader = PyPDFLoader(str(pdf_path))
#     loaded = loader.load()
#     for doc in loaded:
#         doc.metadata["source"] = pdf_path.name
#     pdf_docs.extend(loaded)
#
# pdf_splits = splitter.split_documents(pdf_docs)
# pdf_vectorstore = Chroma.from_documents(
#     documents=pdf_splits,
#     embedding=embeddings,
#     collection_name="mis_pdfs",
#     persist_directory="./chroma_pdfs",
# )
#
# print("PDF chunks:", len(pdf_splits))


## Resumen

Con este patrón ya se puede construir un chatbot básico sobre documentos propios:

```text
loader -> splitter -> embeddings -> vector store -> retriever -> prompt -> LLM
```

A partir de acá, las mejoras típicas son:

- cargar PDFs, HTML, Markdown o bases de datos;
- mejorar chunking;
- agregar reranking;
- agregar permisos por usuario;
- guardar historial de conversación;
- evaluar retrieval y calidad de respuesta.
